In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import joblib
import warnings
warnings.filterwarnings('ignore')

print("🤖 Model Training - Customer Churn Prediction")
print("============================================")


🤖 Model Training - Customer Churn Prediction


In [8]:
df = pd.read_csv('../Data/Processed/churn_ml_dataset.csv')
print(f"📊 Data loaded: {df.shape}")


📊 Data loaded: (7032, 26)


In [9]:
X = df.drop(['Customer_ID', 'Target'], axis=1)
y = df['Target']

# Handle categorical variables
categorical_cols = X.select_dtypes(include=['object', 'bool']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"Features after encoding: {X_encoded.shape[1]}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Features after encoding: 29
Training set: (5625, 29)
Testing set: (1407, 29)


In [10]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = {}
print("\n🎯 Training Models...")

for name, model in models.items():
    print(f"\n--- Training {name} ---")
    
    # Use scaled data for Logistic Regression
    if name == 'Logistic Regression':
        X_tr = X_train_scaled
        X_te = X_test_scaled
    else:
        X_tr = X_train
        X_te = X_test
    
    # Train model
    model.fit(X_tr, y_train)
    
    # Predictions
    y_pred = model.predict(X_te)
    y_pred_proba = model.predict_proba(X_te)[:, 1]
    
    # Calculate metrics
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba),
        'model': model
    }
    
    print(f"✅ {name} - Accuracy: {results[name]['accuracy']:.4f}")

from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

grid_search = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=1000),
    param_grid, cv=5, scoring='roc_auc'
)
grid_search.fit(X_train_scaled, y_train)
print(f"Best params: {grid_search.best_params_}")



🎯 Training Models...

--- Training Logistic Regression ---
✅ Logistic Regression - Accuracy: 0.9346

--- Training Random Forest ---
✅ Random Forest - Accuracy: 0.9339

--- Training XGBoost ---
✅ XGBoost - Accuracy: 0.9289
Best params: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}


In [11]:
results_df = pd.DataFrame(results).T
print("\n📊 Model Performance Comparison:")
display(results_df.round(4))



📊 Model Performance Comparison:


,accuracy,precision,recall,f1,roc_auc,model
Logistic Regression,0.934613,0.887363,0.863636,0.875339,0.97936,"LogisticRegression(max_iter=1000, random_state..."
Random Forest,0.933902,0.880759,0.868984,0.874832,0.975947,"(DecisionTreeClassifier(max_features='sqrt', r..."
XGBoost,0.928927,0.87027,0.860963,0.865591,0.974911,"XGBClassifier(base_score=None, booster=None, c..."


In [12]:
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']

joblib.dump(best_model, '../Models/churn_prediction_model.pkl')
joblib.dump(scaler, '../Models/scaler.pkl')
joblib.dump(X_encoded.columns, '../Models/feature_columns.pkl')

print(f"\n💾 Best model saved: {best_model_name}")
print("📁 Models saved in ../Models/ folder")

print("\n🎯 MODEL TRAINING COMPLETED!")


💾 Best model saved: Logistic Regression
📁 Models saved in ../Models/ folder

🎯 MODEL TRAINING COMPLETED!
